# Base policy analysis

Per-step eval traces of one wandb group's runs (the parquet `scripts/eval.py --run ...` uploads to the
run's files, e.g. with `configs/eval/one_leg_phases.yaml`, which sets `task.cfg.furniture.track_phases`).

Cells: 1 imports, 2 variables, 3 download, 4 preprocessing, then one plot or table per cell.
With more than one run in the group, the plots show the mean over runs with a 95% confidence interval
(Student t over the per-run rates).

In [18]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import wandb
from scipy import stats
import plotly.io as pio; pio.renderers.default = "notebook_connected"

# the repo root, so the phase ids and colours come from the one_leg overlay itself
REPO = Path.cwd().resolve()
while not (REPO / "my_project").is_dir():
    if REPO.parent == REPO:
        raise RuntimeError("run this notebook from inside the hybrid_demos repo")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from my_project.overlays import PHASE_COLORS, PHASE_NAMES  # noqa: E402

In [19]:
# the runs: every run in this wandb group
ENTITY = "hur"
PROJECT = "hybrid_demo_furniture"
GROUP = "physic_aligned_evalHappy"

# which eval of each run: the eval config's stem (the parquet is <stem>.parquet in the run's
# eval/<stem>_<timestamp>/ or videos/<stem>_<timestamp>/ folder); a run with several such evals
# uses its latest
EVAL_STEM = "one_leg_phases"

# where the parquets are downloaded to (git-ignored); a file already there is not downloaded again
CACHE_DIR = REPO / "data_analysis" / "cache"

# the per-step phase column (the env's eval_state channel 'phase/id')
PHASE_COLUMN = "eval_state/phase/id"

# True: timeouts are counted as failures (one 'Failure' node); False: a separate 'Timeout' node
MERGE_TIMEOUT_INTO_FAILURE = True

# the confidence level of the across-run intervals
CONFIDENCE = 0.95

all_eps = "Initialization"

In [20]:
api = wandb.Api()
runs = list(api.runs(f"{ENTITY}/{PROJECT}", filters={"group": GROUP}))
if not runs:
    raise LookupError(f"no runs in {ENTITY}/{PROJECT} group {GROUP!r}")

trace_name = re.compile(rf"^(eval|videos)/{re.escape(EVAL_STEM)}_(\d{{8}}-\d{{6}})/{re.escape(EVAL_STEM)}\.parquet$")

frames, missing = [], []
for run in runs:
    found = [(m.group(2), f) for f in run.files() if (m := trace_name.match(f.name))]
    if not found:
        missing.append(run.name)
        continue
    stamp, handle = max(found, key=lambda x: x[0])  # the latest eval
    root = CACHE_DIR / ENTITY / PROJECT / GROUP / run.id
    path = root / handle.name
    if not path.exists():
        handle.download(root=str(root), replace=True)
    frame = pd.read_parquet(path)
    frame.insert(0, "run", run.name)
    frame.insert(1, "eval", f"{EVAL_STEM}_{stamp}")
    frames.append(frame)
    print(f"{run.name} ({run.id}): {handle.name}, {len(frame)} rows")

if missing:
    print(f"no {EVAL_STEM} eval in {len(missing)} of {len(runs)} runs, left out: {missing}")
if not frames:
    raise FileNotFoundError(f"none of the {len(runs)} runs in group {GROUP!r} has a {EVAL_STEM} eval trace")
steps = pd.concat(frames, ignore_index=True)
print(f"{steps['run'].nunique()} runs, {len(steps)} steps")

physic_aligned_evalHappy_a0 (1vs2w5ji): eval/one_leg_phases_20261021-114409/one_leg_phases.parquet, 407650 rows
1 runs, 407650 steps


In [21]:
# phase ids (the env's furniture.track_phases): 0-3 the task phases, 4-6 how the episode ended (its last step)
TASK_PHASES = [name.capitalize() for name in PHASE_NAMES[:4]]  # Corner, Grasp, Insert, Screw
SUCCESS, FAILURE, TIMEOUT = 4, 5, 6

if PHASE_COLUMN not in steps:
    raise KeyError(
        f"the traces have no {PHASE_COLUMN!r} column: eval with task.cfg.furniture.track_phases: true "
        f"(configs/eval/one_leg_phases.yaml)"
    )
if steps[PHASE_COLUMN].isna().any():
    raise ValueError(f"{PHASE_COLUMN} has NaN rows: some rounds were evaluated without phase tracking")

# one episode per (run, round, env): each env's first episode of the round
steps = steps.sort_values(["run", "round", "env", "step"])
steps["phase"] = steps[PHASE_COLUMN].round().astype(int)
grouped = steps.groupby(["run", "round", "env"], sort=False)["phase"]
task = lambda p: p[p < SUCCESS]  # noqa: E731  the task-phase steps (the ending step is 4-6)
episodes = pd.DataFrame({
    "last": grouped.last(),
    # the phase id is the CURRENT phase: the one the episode ended in was not completed
    "ended_in": grouped.apply(lambda p: task(p).iloc[-1] if len(task(p)) else 0),
    "went_back": grouped.apply(lambda p: bool((np.diff(task(p).to_numpy()) < 0).any())),
})
if (episodes["last"] < SUCCESS).any():
    bad = episodes[episodes["last"] < SUCCESS]
    raise ValueError(f"{len(bad)} episodes have no ending phase (4-6) on their last step:\n{bad.head()}")
if episodes["went_back"].any():
    print(f"note: {int(episodes['went_back'].sum())} episodes went back to an earlier phase; "
          "each is counted by the phase it ended in")

# task phases completed: those before the one it ended in; a success completed all of them
episodes["completed"] = np.where(episodes["last"] == SUCCESS, len(TASK_PHASES), episodes["ended_in"])
outcome_names = {SUCCESS: "Success", FAILURE: "Failure", TIMEOUT: "Failure" if MERGE_TIMEOUT_INTO_FAILURE else "Timeout"}
episodes["outcome"] = episodes["last"].map(outcome_names)
episodes = episodes.reset_index()

# per run: episodes completing each task phase, ending in each outcome, and the Sankey's drops:
# '<phase>-><outcome>' = completed the phase before <phase> (or nothing, for Corner) but not
# <phase>, ended as <outcome>
OUTCOMES = ["Success", "Failure"] + ([] if MERGE_TIMEOUT_INTO_FAILURE else ["Timeout"])


def run_rates(ep):
    row = {"episodes": len(ep)}
    for i, name in enumerate(TASK_PHASES):
        row[name] = (ep["completed"] > i).sum()
    for name in OUTCOMES:
        row[name] = (ep["outcome"] == name).sum()
    for i, name in enumerate(TASK_PHASES):
        stopped = ep[ep["completed"] == i]
        for out in OUTCOMES:
            row[f"{name}->{out}"] = (stopped["outcome"] == out).sum()
    return pd.Series(row)


counts = episodes.groupby("run").apply(run_rates)
rates = counts.drop(columns="episodes").div(counts["episodes"], axis=0)
n_runs = len(rates)


def interval(column):
    """The mean over runs and its CONFIDENCE half-width (Student t; 0 with one run)."""
    x = rates[column].to_numpy(float)
    if n_runs < 2:
        return x.mean(), 0.0
    return x.mean(), stats.t.ppf(0.5 + CONFIDENCE / 2, n_runs - 1) * x.std(ddof=1) / np.sqrt(n_runs)


summary = pd.DataFrame(
    [(c, *interval(c), counts[c].sum(), counts["episodes"].sum()) for c in TASK_PHASES + OUTCOMES],
    columns=["node", "mean", "ci_half_width", "episodes_pooled", "of_pooled"],
).set_index("node")
display(counts)
display(summary.style.format({"mean": "{:.1%}", "ci_half_width": "{:.1%}"}))

,episodes,Corner,Grasp,Insert,Screw,Success,Failure,Corner->Success,Corner->Failure,Grasp->Success,Grasp->Failure,Insert->Success,Insert->Failure,Screw->Success,Screw->Failure
run,,,,,,,,,,,,,,,
physic_aligned_evalHappy_a0,512,473,460,365,280,280,232,0,39,0,13,0,95,0,85


,mean,ci_half_width,episodes_pooled,of_pooled
node,,,,
Corner,92.4%,0.0%,473,512
Grasp,89.8%,0.0%,460,512
Insert,71.3%,0.0%,365,512
Screw,54.7%,0.0%,280,512
Success,54.7%,0.0%,280,512
Failure,45.3%,0.0%,232,512


In [22]:
# phase progression: a phase's node is the episodes that COMPLETED it; from it, those that also
# completed the next phase go on and those that did not drop to how they ended. Episodes that did
# not complete Corner start from an 'All episodes' node (shown only when there are any).
# Widths are the mean fraction of episodes over the group's runs, so every node's in- and outflow match.
needs_start = any(interval(f"{TASK_PHASES[0]}->{o}")[0] > 0 for o in OUTCOMES)
nodes = ([all_eps] if needs_start else []) + TASK_PHASES + OUTCOMES
index = {name: i for i, name in enumerate(nodes)}
rgb = lambda c, a=1.0: f"rgba({c[0]},{c[1]},{c[2]},{a})"  # noqa: E731
node_colour = {name: PHASE_COLORS[i] for i, name in enumerate(TASK_PHASES)}
node_colour.update({all_eps: (150, 150, 150), "Success": PHASE_COLORS[SUCCESS],
                    "Failure": PHASE_COLORS[FAILURE], "Timeout": PHASE_COLORS[TIMEOUT]})


def label(name):
    total = int(counts["episodes"].sum())
    if name == all_eps:
        return f"{name} ({total})"
    pooled = int(summary.loc[name, "episodes_pooled"])
    if n_runs == 1:
        return f"{name}<br>{pooled}/{total}"
    mean, half = summary.loc[name, ["mean", "ci_half_width"]]
    return f"{name}<br>{mean:.0%} ± {half:.0%}<br>({pooled}/{total})"


source, target, value, colour, hover = [], [], [], [], []


def link(a, b, column):
    mean, half = interval(column)
    if mean <= 0:
        return
    source.append(index[a]); target.append(index[b]); value.append(mean)
    colour.append(rgb(node_colour[b], 0.35))
    ci = "" if n_runs == 1 else f" ± {half:.1%}"
    hover.append(f"{a} → {b}: {mean:.1%}{ci} ({int(counts[column].sum())} episodes)")


# into each task phase from the one before it, and the drops: completed the one before, not this one
for i, name in enumerate(TASK_PHASES):
    before = TASK_PHASES[i - 1] if i else all_eps
    if before in index:
        link(before, name, name)
    for out in OUTCOMES:
        if before in index:
            link(before, out, f"{name}->{out}")
# completing the last phase is the task's success
link(TASK_PHASES[-1], "Success", "Success")

# every phase node's inflow must equal its outflow and its label, or the picture lies
for name in TASK_PHASES:
    i = index[name]
    inflow = sum(v for s_, t, v in zip(source, target, value) if t == i)
    outflow = sum(v for s_, t, v in zip(source, target, value) if s_ == i)
    expected = summary.loc[name, "mean"]
    has_source = name != TASK_PHASES[0] or needs_start  # Corner has no inflow without the 'All episodes' node
    if not ((np.isclose(inflow, expected) or not has_source) and np.isclose(outflow, expected)):
        raise AssertionError(f"{name}: inflow {inflow:.3f}, outflow {outflow:.3f}, label {expected:.3f}")

fig = go.Figure(go.Sankey(
    # plotly's own layout: pinning node positions (x, y) leaves its link order from the unpinned
    # layout, which crosses the drops over the forward flows
    arrangement="snap",
    node=dict(label=[label(n) for n in nodes], color=[rgb(node_colour[n]) for n in nodes], pad=25, thickness=18),
    link=dict(source=source, target=target, value=value, color=colour, customdata=hover, hovertemplate="%{customdata}<extra></extra>"),
))
ci_note = "" if n_runs == 1 else f", mean ± {CONFIDENCE:.0%} CI over {n_runs} runs"
fig.update_layout(title=f"{GROUP}: phase progression ({EVAL_STEM}{ci_note})", font_size=13, width=1200, height=500)
fig.show()